In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src" / "clases_po").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

from clases_po import HFDatasetLoader

/home/runner/work/ML_202604_Santiago_Lopez_F/ML_202604_Santiago_Lopez_F/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
hfdataset= HFDatasetLoader(dataset_name="aptologia/precios-arriendo-colombia", split="train", filename="datos/precios-arriendo-colombia.csv")

In [3]:
df = hfdataset.load()

Generating train split:   0%|          | 0/1460 [00:00<?, ? examples/s]

Generating train split: 100%|██████████| 1460/1460 [00:00<00:00, 184076.83 examples/s]

In [4]:
df.head(15)

,fecha,ciudad_id,ciudad,barrio,avisos,p25,mediana,p75,con_parqueadero,con_telefono
0,2026-10-05,medellin,Medellín,El Poblado,17124,4400000,5700000,7700000,15907,6599
1,2026-10-05,medellin,Medellín,Poblado,7593,4100000,5650000,8000000,6509,841
2,2026-10-05,medellin,Medellín,Laureles,5780,3300000,4200000,5300000,4283,2169
3,2026-10-05,medellin,Medellín,Castropol,2730,4200000,5350000,7000000,2286,1547
4,2026-10-05,medellin,Medellín,El Tesoro,2471,5800000,7700000,11500000,2167,1278
5,2026-10-05,medellin,Medellín,Santa Maria de los Angeles,2237,4100000,5600000,6800000,1930,922
6,2026-10-05,medellin,Medellín,Calasanz,2071,2200000,2600000,3200000,1175,742
7,2026-10-05,medellin,Medellín,Los Balsos,1870,4900000,6175000,9000000,1490,645
8,2026-10-05,medellin,Medellín,Las Palmas,1808,3900000,4300000,6200000,1503,659
9,2026-10-05,medellin,Medellín,Ciudad del Rio,1740,4000000,4500000,5412500,1372,900


In [5]:
import unicodedata
 
def normalizar(texto):
    texto = str(texto).strip().lower()
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(c for c in texto if not unicodedata.combining(c))  # quita tildes
    return " ".join(texto.split())  # colapsa espacios dobles
df["barrio_norm"] = df["barrio"].apply(normalizar)

df.head(15)

,fecha,ciudad_id,ciudad,barrio,avisos,p25,mediana,p75,con_parqueadero,con_telefono,barrio_norm
0,2026-10-05,medellin,Medellín,El Poblado,17124,4400000,5700000,7700000,15907,6599,el poblado
1,2026-10-05,medellin,Medellín,Poblado,7593,4100000,5650000,8000000,6509,841,poblado
2,2026-10-05,medellin,Medellín,Laureles,5780,3300000,4200000,5300000,4283,2169,laureles
3,2026-10-05,medellin,Medellín,Castropol,2730,4200000,5350000,7000000,2286,1547,castropol
4,2026-10-05,medellin,Medellín,El Tesoro,2471,5800000,7700000,11500000,2167,1278,el tesoro
5,2026-10-05,medellin,Medellín,Santa Maria de los Angeles,2237,4100000,5600000,6800000,1930,922,santa maria de los angeles
6,2026-10-05,medellin,Medellín,Calasanz,2071,2200000,2600000,3200000,1175,742,calasanz
7,2026-10-05,medellin,Medellín,Los Balsos,1870,4900000,6175000,9000000,1490,645,los balsos
8,2026-10-05,medellin,Medellín,Las Palmas,1808,3900000,4300000,6200000,1503,659,las palmas
9,2026-10-05,medellin,Medellín,Ciudad del Rio,1740,4000000,4500000,5412500,1372,900,ciudad del rio


In [6]:
#Reorganiza las filas del DataFrame para mostrar los precios más altos al principio, utilizando la columna mediana
df_transformado = df.sort_values(by='mediana', ascending=False)
df_transformado.head(5)

,fecha,ciudad_id,ciudad,barrio,avisos,p25,mediana,p75,con_parqueadero,con_telefono,barrio_norm
806,2026-10-05,bogota,Bogotá,Cerros de los Alpes,24,17000000,20500000,25000000,24,11,cerros de los alpes
883,2026-10-05,bogota,Bogotá,La Cabrera Chico,17,11000000,18000000,19873000,15,9,la cabrera chico
894,2026-10-05,bogota,Bogotá,Bosques del Marquez,16,14392000,17000000,17000000,16,7,bosques del marquez
807,2026-10-05,bogota,Bogotá,Bosques del Marques,24,14392000,15800000,18000000,21,4,bosques del marques
538,2026-10-05,bogota,Bogotá,Cabrera,252,10500000,15000000,18000000,217,118,cabrera


In [7]:
#Filtra el DataFrame para conservar únicamente las columnas más relevantes para tu análisis.
df_transformado = df[['fecha', 'ciudad', 'barrio_norm', 'mediana', 'con_parqueadero']]
df_transformado.head(5)

,fecha,ciudad,barrio_norm,mediana,con_parqueadero
0,2026-10-05,Medellín,el poblado,5700000,15907
1,2026-10-05,Medellín,poblado,5650000,6509
2,2026-10-05,Medellín,laureles,4200000,4283
3,2026-10-05,Medellín,castropol,5350000,2286
4,2026-10-05,Medellín,el tesoro,7700000,2167


In [8]:
#Filtrado de filas por condición numérica
df_transformado = df[df['avisos'] > 10]
df_transformado.head(5)

,fecha,ciudad_id,ciudad,barrio,avisos,p25,mediana,p75,con_parqueadero,con_telefono,barrio_norm
0,2026-10-05,medellin,Medellín,El Poblado,17124,4400000,5700000,7700000,15907,6599,el poblado
1,2026-10-05,medellin,Medellín,Poblado,7593,4100000,5650000,8000000,6509,841,poblado
2,2026-10-05,medellin,Medellín,Laureles,5780,3300000,4200000,5300000,4283,2169,laureles
3,2026-10-05,medellin,Medellín,Castropol,2730,4200000,5350000,7000000,2286,1547,castropol
4,2026-10-05,medellin,Medellín,El Tesoro,2471,5800000,7700000,11500000,2167,1278,el tesoro


In [9]:
from clases_po import Reportes

In [10]:
reportes = Reportes()
texto = reportes.generar_reporte(df)

print(texto)

REPORTE DEL DATAFRAME
Filas: 1460
Columnas: 11
Duplicados: 0

Columnas y tipos:
- fecha: object
- ciudad_id: object
- ciudad: object
- barrio: object
- avisos: int64
- p25: int64
- mediana: int64
- p75: int64
- con_parqueadero: int64
- con_telefono: int64
- barrio_norm: object

Valores nulos por columna:
- fecha: 0
- ciudad_id: 0
- ciudad: 0
- barrio: 0
- avisos: 0
- p25: 0
- mediana: 0
- p75: 0
- con_parqueadero: 0
- con_telefono: 0
- barrio_norm: 0

Estadísticas numéricas:
          avisos           p25       mediana           p75  con_parqueadero  con_telefono
count   1460.000  1.460000e+03  1.460000e+03  1.460000e+03         1460.000      1460.000
mean     147.713  2.444034e+06  3.084563e+06  4.002385e+06           98.788        52.418
std      595.177  1.656497e+06  2.281845e+06  3.211595e+06          497.104       213.097
min       15.000  4.000000e+05  4.000000e+05  4.000000e+05            0.000         0.000
25%       23.000  1.400000e+06  1.600000e+06  1.950000e+06           1

In [11]:
ruta = reportes.guardar_reporte(
    df,
    "reportes/precios-arriendo-colombia.txt",
)

print(f"Reporte guardado en: {ruta}")

Reporte guardado en: reportes/precios-arriendo-colombia.txt


In [12]:
df["barrio"].unique()          # array con los nombres únicos
df["barrio"].nunique()         # cuántos hay
df["barrio"].value_counts()    # cuántas filas tiene cada uno

barrio
Centro         8
Bellavista     5
El Porvenir    5
La Floresta    5
El Bosque      4
              ..
La Camelia     1
La Francia     1
La Enea        1
Palogrande     1
Poblado        1
Name: count, Length: 1300, dtype: int64

In [13]:
print(df["barrio"].unique())
print(df["barrio"].nunique())

['El Poblado' 'Poblado' 'Laureles' ... 'Ipanema' 'Neiva' 'San Jorge']
1300


In [14]:
print(df["barrio_norm"].unique())
print(df["barrio_norm"].nunique())

['el poblado' 'poblado' 'laureles' ... 'ipanema' 'neiva' 'san jorge']
1300


In [15]:
df[df["barrio"] == "Centro"]

,fecha,ciudad_id,ciudad,barrio,avisos,p25,mediana,p75,con_parqueadero,con_telefono,barrio_norm
96,2026-10-05,medellin,Medellín,Centro,246,1850000,2500000,3100000,81,142,centro
818,2026-10-05,bogota,Bogotá,Centro,23,2100000,2290000,2550000,10,9,centro
1116,2026-10-05,rionegro,Rionegro,Centro,463,1695788,2316847,2795788,337,183,centro
1247,2026-10-05,bucaramanga,Bucaramanga,Centro,79,1200000,1500000,1950000,58,21,centro
1315,2026-10-05,cucuta,Cúcuta,Centro,45,800000,900000,1250000,17,6,centro
1365,2026-10-05,chia,Chía,Centro,81,1350000,1700000,1980000,60,27,centro
1393,2026-10-05,pereira,Pereira,Centro,355,1950000,2700000,3700000,211,2,centro
1422,2026-10-05,manizales,Manizales,Centro,51,1200000,1500000,2700000,20,1,centro
